In [19]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [20]:
import os
import sys
from pathlib import Path
import glob

project_root = Path.cwd().parent
sys.path.append(str(project_root))

import pandas as pd
import numpy as np

from src.load_data import load_confocal, load_confocal2

# Experimental conditions

In [21]:
experimental_path = Path.cwd().parent / 'data/silver_layer/experimental_results.pkl'
df_experimental = pd.read_pickle(experimental_path)

# Confocal

In [22]:
data_path = Path.cwd().parent / 'data/bronze_layer/confocal'

In [23]:
z1 = load_confocal(data_path, '2024_02_27', 12)
z2 = load_confocal(data_path, '2024_02_28', 12)
z3 = load_confocal(data_path, '2024_03_04', 12)
z4 = load_confocal(data_path, '2024_03_05', 12)
z5 = load_confocal(data_path, '2024_03_06', 12)
z6 = load_confocal(data_path, '2024_03_12', 12)
z7 = load_confocal(data_path, '2024_03_13', 12)
z8 = load_confocal(data_path, '2024_03_14', 12)
z9 = load_confocal(data_path, '2024_03_16', 14)
z10 = load_confocal(data_path, '2024_03_18', 14)
z11 = load_confocal(data_path, '2024_03_19', 14)
z12 = load_confocal(data_path, '2024_03_21', 14)
z13 = load_confocal(data_path, '2024_03_22', 14)
z14 = load_confocal(data_path, '2024_03_22_v2', 14)
z15 = load_confocal(data_path, '2024_03_23', 14)
z16 = load_confocal(data_path, '2024_03_25', 14)
z17 = load_confocal(data_path, '2024_03_25_v2', 14)
z18 = load_confocal(data_path, '2024_03_25_v3', 14)
z19 = load_confocal(data_path, '2024_03_26', 14)

In [24]:
z = []

z.extend([z1, z2, z3, z4, z5, z6, z7, z8, z9, z10, z11, z12, z13, z14, z15, z16, z17, z18, z19])

In [25]:
columns = ['t1_mean', 't1_median', 't1_std', 't1_missing_percentage', 't2_mean', 't2_median', 't2_std', 't2_missing_percentage', 'df', 'number', 'delta_time']

In [ ]:
T1_THRESHOLD = 0.985
N_WATER = 1.333
N_BK7 = 1.519
MIN_T2 = 0.373
T2_THRESHOLDS = [8.02, 8.03, 8.04, 8.05, 8.10, 8.20, 8.50]

rows = []
run_id = 1

for j, zn in enumerate(z):

    for i, run_df in enumerate(zn):

        delta_time = (
            run_df['time'].diff().mean()
            if len(run_df) > 1 else np.nan
        )

        # Corrected t2 (d2 redistribution, no cap) for threshold statistics
        d1 = (run_df['t1'] - T1_THRESHOLD).clip(lower=0)
        d2 = d1 * (N_WATER / N_BK7)

        t2_corr = run_df['t2'].copy()
        mask_present = t2_corr.notna()
        t2_corr.loc[mask_present] = t2_corr.loc[mask_present] + d2[mask_present]
        mask_fill = t2_corr.isna() & (d2 > MIN_T2)
        t2_corr.loc[mask_fill] = d2[mask_fill]

        n_total = len(run_df)
        n_valid_t2 = int(t2_corr.notna().sum())

        row = {

            'run_id': run_id,

            't1_mean': round(run_df['t1'].mean(), 3),
            't1_median': round(run_df['t1'].median(), 3),
            't1_std': round(run_df['t1'].std(), 3),

            't1_missing_percentage':
                round(
                    100 - run_df['t1'].count()
                    / run_df['time'].count() * 100,
                    1
                ),

            't2_mean': round(run_df['t2'].mean(), 3),
            't2_median': round(run_df['t2'].median(), 3),
            't2_std': round(run_df['t2'].std(), 3),
            't2_max': round(run_df['t2'].max(), 3),

            't2_missing_percentage':
                round(
                    100 - run_df['t2'].count()
                    / run_df['time'].count() * 100,
                    1
                ),

            'delta_time': delta_time,
        }

        for thr in T2_THRESHOLDS:
            n_above = int((t2_corr > thr).sum())
            col = f'{thr:.2f}'.replace('.', '')
            row[f't2_above_{col}_pct_valid'] = (
                round(100 * n_above / n_valid_t2, 3) if n_valid_t2 > 0 else np.nan
            )
            row[f't2_above_{col}_pct_abs'] = round(100 * n_above / n_total, 3)

        rows.append(row)

        run_id += 1

df = pd.DataFrame(rows)

df['run_id'] = df['run_id'].astype(int)

In [27]:
save_path = Path.cwd().parent / 'data/silver_layer'
save_path.mkdir(parents=True, exist_ok=True)

df.to_pickle(
    save_path / 'confocal_results.pkl',
)

df.to_csv(
    save_path / 'confocal_results.csv'
)

In [28]:
save_path = Path.cwd().parent / 'data/silver_layer/confocal_runs'

save_path.mkdir(exist_ok=True)

run_id = 1

for zn in z:

    for run_df in zn:

        filename = save_path / f"confocal_run_{run_id}.csv"

        run_df.to_csv(filename, index=False)

        run_id += 1

# Process tube thickness

In [ ]:
T1_THRESHOLD = 0.985
N_WATER = 1.333
N_BK7 = 1.519
MIN_T2 = 0.373
T2_CAP = 8.02

source_path = Path.cwd().parent / 'data/silver_layer/confocal_runs'
processed_path = Path.cwd().parent / 'data/silver_layer/confocal_runs_processed'
processed_path.mkdir(exist_ok=True)

files = sorted(
    source_path.glob('confocal_run_*.csv'),
    key=lambda x: int(x.stem.split('_')[-1])
)

for file in files:
    run_df = pd.read_csv(file)

    d1 = (run_df['t1'] - T1_THRESHOLD).clip(lower=0)
    d2 = d1 * (N_WATER / N_BK7)

    run_df['t1'] = run_df['t1'] - d1

    mask_present = run_df['t2'].notna()
    run_df.loc[mask_present, 't2'] = run_df.loc[mask_present, 't2'] + d2[mask_present]

    mask_fill = run_df['t2'].isna() & (d2 > MIN_T2)
    run_df.loc[mask_fill, 't2'] = d2[mask_fill]

    run_df['t2'] = run_df['t2'].clip(upper=T2_CAP)

    run_df.to_csv(processed_path / file.name, index=False)

In [30]:
confocal_path = (
    Path.cwd().parent
    / "data/silver_layer/confocal_runs_processed"
)

rows = []

# Sort by run number if files are named confocal_run_1.csv, confocal_run_2.csv, ...
files = sorted(
    confocal_path.glob("confocal_run_*.csv"),
    key=lambda x: int(x.stem.split("_")[-1])
)

for run_id, file in enumerate(files, start=1):

    run_df = pd.read_csv(file)

    delta_time = (
        run_df["time"].diff().mean()
        if len(run_df) > 1 else np.nan
    )

    rows.append({

        "run_id": run_id,

        "t1_mean": round(run_df["t1"].mean(), 3),
        "t1_median": round(run_df["t1"].median(), 3),
        "t1_std": round(run_df["t1"].std(), 3),

        "t1_missing_percentage":
            round(
                100
                - run_df["t1"].count()
                / run_df["time"].count() * 100,
                1
            ),

        "t2_mean": round(run_df["t2"].mean(), 3),
        "t2_median": round(run_df["t2"].median(), 3),
        "t2_std": round(run_df["t2"].std(), 3),

        "t2_missing_percentage":
            round(
                100
                - run_df["t2"].count()
                / run_df["time"].count() * 100,
                1
            ),

        "delta_time": delta_time,
    })

df = pd.DataFrame(rows)

df["run_id"] = df["run_id"].astype(int)

In [ ]:
save_path = Path.cwd().parent / 'data/silver_layer'
save_path.mkdir(parents=True, exist_ok=True)

df.to_pickle(
    save_path / 'confocal_results_processed.pkl',
)

df.to_csv(
    save_path / 'confocal_results_processed.csv'
)

# Merge experimental

In [32]:
df_combined = pd.merge(df, df_experimental[['run_id', 'jl', 'jg', 'date_identifier', 'pattern', 'reduced_pattern']], on='run_id', how='left')

In [33]:
pd.read_pickle(save_path/'confocal_results_processed.pkl')['t2_mean'].sum()

np.float64(346.84799999999996)

In [34]:
pd.read_pickle(save_path/'confocal_results.pkl')['t2_mean'].sum()

np.float64(347.508)